# Hikvision · Consulta y reportes — Guía en 13 pasos

Este cuaderno prueba cada parte por separado (entorno, conectividad, autenticación, consulta de equipo, eventos paginados, manejo de errores) antes de generar los informes JSON y HTML.

**Supuesto:** equipo de control de acceso Hikvision con **ISAPI** por HTTP/HTTPS en tu red local (endpoint `AcsEvent`). Si tu equipo es un NVR/DVR de cámaras, cambia el endpoint del Paso 9 por `ContentMgmt/search`.

**Contraseña:** se pide con `getpass` en el Paso 6 y solo vive en memoria mientras el kernel esté activo. No se escribe en ningún archivo.

## Paso 1 — Verificar que el kernel usa el `.venv` correcto

Antes de instalar o importar nada, confirma que Jupyter está corriendo con el Python del entorno virtual (no con tu Python global).

In [1]:
import sys
print('Ejecutable de Python en uso:')
print(sys.executable)
print()
if '.venv' in sys.executable:
    print('OK: el kernel apunta al entorno virtual .venv')
else:
    print('ATENCIÓN: este kernel NO parece ser el de .venv. '
          'Cambia el kernel desde el menú Kernel > Change Kernel.')

Ejecutable de Python en uso:
c:\Users\izipa\Downloads\hikvision_paso_a_paso\.venv\Scripts\python.exe

OK: el kernel apunta al entorno virtual .venv


## Paso 2 — Verificar que las dependencias están instaladas

Si ya corriste `pip install -r requirements.txt` dentro del `.venv`, esta celda debe mostrar las versiones sin error.

In [2]:
import importlib

for paquete in ('requests', 'urllib3'):
    try:
        mod = importlib.import_module(paquete)
        print(f'{paquete}: OK (versión {mod.__version__})')
    except ImportError:
        print(f'{paquete}: FALTA. Corre: .\\.venv\\Scripts\\python.exe -m pip install -r requirements.txt')

requests: OK (versión 2.34.2)
urllib3: OK (versión 2.7.0)


## Paso 3 — Imports generales

In [3]:
import json
import socket
import getpass
from datetime import datetime, timedelta
from pathlib import Path

import requests
from requests.auth import HTTPDigestAuth
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
print('Imports listos.')

Imports listos.


## Paso 4 — Configuración de conexión

Ajusta estos valores según tu equipo. Todavía no se hace ninguna conexión en este paso.

In [ ]:
HOST = '192.168.18.211'   # IP o hostname del terminal
PORT = 80                # 80 para HTTP, 443 (típico) para HTTPS
USE_HTTPS = False
USUARIO = 'admin'
TIMEOUT = 8               # segundos, para todas las peticiones

PROTOCOLO = 'https' if USE_HTTPS else 'http'
BASE_URL = f'{PROTOCOLO}://{HOST}:{PORT}'
print('Base URL configurada:', BASE_URL)

Base URL configurada: http://192.168.1.64:80


## Paso 5 — Prueba de conectividad de red (sin credenciales)

Antes de autenticarse, probamos si el puerto responde a nivel de red. Esto detecta problemas de IP/firewall/VPN antes de meter la contraseña.

In [5]:
def probar_puerto(host, port, timeout=5):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True, None
    except OSError as e:
        return False, str(e)

ok, error = probar_puerto(HOST, PORT, TIMEOUT)
if ok:
    print(f'OK: {HOST}:{PORT} responde a nivel de red.')
else:
    print(f'NO se pudo conectar a {HOST}:{PORT} -> {error}')
    print('Revisa: misma red/VPN que el equipo, IP y puerto correctos, firewall.')

NO se pudo conectar a 192.168.1.64:80 -> timed out
Revisa: misma red/VPN que el equipo, IP y puerto correctos, firewall.


## Paso 6 — Ingresar la contraseña y crear la sesión autenticada

La contraseña se pide con `getpass` (no se muestra en pantalla). Se guarda solo en la variable `PASSWORD`, en memoria, mientras el kernel esté vivo.

In [6]:
PASSWORD = getpass.getpass('Contraseña del terminal Hikvision: ')

session = requests.Session()
session.auth = HTTPDigestAuth(USUARIO, PASSWORD)
session.verify = False  # certificados autofirmados típicos en LAN
print('Sesión creada. Credenciales en memoria, no en disco.')

Sesión creada. Credenciales en memoria, no en disco.


## Paso 7 — Consultar información del equipo (device info)

Endpoint: `GET /ISAPI/System/deviceInfo?format=json`. Esta es la primera llamada real autenticada — si falla con `401`, revisa usuario/contraseña; si da timeout, revisa el Paso 5.

In [7]:
def get_device_info(session, base_url, timeout=TIMEOUT):
    url = f'{base_url}/ISAPI/System/deviceInfo?format=json'
    resp = session.get(url, timeout=timeout)
    resp.raise_for_status()
    try:
        return resp.json()
    except ValueError:
        return {'raw_xml': resp.text}

device_info = None
try:
    device_info = get_device_info(session, BASE_URL)
    print('Información del equipo obtenida:')
    print(json.dumps(device_info, indent=2, ensure_ascii=False))
except requests.exceptions.RequestException as e:
    print('Error consultando el equipo:', e)

Error consultando el equipo: HTTPConnectionPool(host='192.168.1.64', port=80): Max retries exceeded with url: /ISAPI/System/deviceInfo?format=json (Caused by ConnectTimeoutError(<HTTPConnection(host='192.168.1.64', port=80) at 0x1f5ae929190>, 'Connection to 192.168.1.64 timed out. (connect timeout=8)'))


## Paso 8 — Probar eventos: solo la primera página (prueba de paginación)

Antes de recorrer todo el historial, probamos una sola página para validar el formato de respuesta y el tamaño del lote.

In [8]:
def pedir_pagina_eventos(session, base_url, start_time, end_time, posicion=0, page_size=30, timeout=TIMEOUT):
    url = f'{base_url}/ISAPI/AccessControl/AcsEvent?format=json'
    fmt = '%Y-%m-%dT%H:%M:%S-05:00'  # ajusta el offset de zona horaria si aplica
    payload = {
        'AcsEventCond': {
            'searchID': '1',
            'searchResultPosition': posicion,
            'maxResults': page_size,
            'major': 0,
            'minor': 0,
            'startTime': start_time.strftime(fmt),
            'endTime': end_time.strftime(fmt),
        }
    }
    resp = session.post(url, json=payload, timeout=timeout)
    resp.raise_for_status()
    return resp.json()

fin = datetime.now()
inicio = fin - timedelta(days=1)

try:
    primera_pagina = pedir_pagina_eventos(session, BASE_URL, inicio, fin, posicion=0)
    bloque = primera_pagina.get('AcsEvent', {})
    print('numOfMatches:', bloque.get('numOfMatches'))
    print('Eventos en esta página:', len(bloque.get('InfoList', [])))
except requests.exceptions.RequestException as e:
    print('Error en la primera página de eventos:', e)

Error en la primera página de eventos: HTTPConnectionPool(host='192.168.1.64', port=80): Max retries exceeded with url: /ISAPI/AccessControl/AcsEvent?format=json (Caused by ConnectTimeoutError(<HTTPConnection(host='192.168.1.64', port=80) at 0x1f5ad5eb5d0>, 'Connection to 192.168.1.64 timed out. (connect timeout=8)'))


## Paso 9 — Recorrer todas las páginas de eventos

Con el formato validado en el Paso 8, ahora recorremos todas las páginas hasta agotar los resultados.

> Si tu equipo es un NVR/DVR de cámaras, cambia la URL de esta función a `POST /ISAPI/ContentMgmt/search?format=json` y ajusta el payload según la documentación ISAPI de tu modelo.

In [9]:
def search_events(session, base_url, start_time, end_time, page_size=30, max_pages=50, timeout=TIMEOUT):
    eventos = []
    posicion = 0
    for pagina in range(max_pages):
        data = pedir_pagina_eventos(session, base_url, start_time, end_time, posicion, page_size, timeout)
        bloque = data.get('AcsEvent', {})
        lote = bloque.get('InfoList', [])
        eventos.extend(lote)

        num_matches = bloque.get('numOfMatches', 0)
        print(f'Página {pagina + 1}: {len(lote)} eventos (posición {posicion})')

        if num_matches < page_size or not lote:
            break
        posicion += page_size
    return eventos

eventos = []
try:
    eventos = search_events(session, BASE_URL, inicio, fin)
    print(f'Total de eventos obtenidos: {len(eventos)}')
except requests.exceptions.RequestException as e:
    print('Error recorriendo eventos:', e)

Error recorriendo eventos: HTTPConnectionPool(host='192.168.1.64', port=80): Max retries exceeded with url: /ISAPI/AccessControl/AcsEvent?format=json (Caused by ConnectTimeoutError(<HTTPConnection(host='192.168.1.64', port=80) at 0x1f5ae93fa90>, 'Connection to 192.168.1.64 timed out. (connect timeout=8)'))


## Paso 10 — Prueba de manejo de errores

Esta celda **no depende de tu equipo real**: prueba que las funciones capturan correctamente errores típicos (host inválido, timeout, credenciales incorrectas), usando un host de prueba que no existe. Es la prueba que se puede validar sin conectarse al terminal.

In [10]:
sesion_prueba = requests.Session()
sesion_prueba.auth = HTTPDigestAuth('usuario_prueba', 'clave_prueba')
sesion_prueba.verify = False

casos = [
    ('Host inexistente', 'http://10.255.255.1:80', 2),
    ('Puerto cerrado (ejemplo)', f'http://{HOST}:9999', 2),
]

for nombre, url_base, tmo in casos:
    try:
        get_device_info(sesion_prueba, url_base, timeout=tmo)
        print(f'{nombre}: respondió (inesperado en este caso de prueba)')
    except requests.exceptions.ConnectTimeout:
        print(f'{nombre}: capturado correctamente -> ConnectTimeout')
    except requests.exceptions.ConnectionError as e:
        print(f'{nombre}: capturado correctamente -> ConnectionError')
    except requests.exceptions.RequestException as e:
        print(f'{nombre}: capturado correctamente -> {type(e).__name__}')

print('\nPrueba de manejo de errores completada (sin usar tu equipo real).')

Host inexistente: capturado correctamente -> ConnectTimeout
Puerto cerrado (ejemplo): capturado correctamente -> ConnectTimeout

Prueba de manejo de errores completada (sin usar tu equipo real).


## Paso 11 — Guardar el JSON con los resultados

In [11]:
salida = {
    'consultado_en': datetime.now().isoformat(),
    'equipo': {'host': HOST, 'puerto': PORT},
    'device_info': device_info,
    'rango': {'inicio': inicio.isoformat(), 'fin': fin.isoformat()},
    'total_eventos': len(eventos),
    'eventos': eventos,
}

Path('output').mkdir(exist_ok=True)
json_path = Path('output/hikvision_reporte.json')
json_path.write_text(json.dumps(salida, indent=2, ensure_ascii=False), encoding='utf-8')
print('JSON guardado en:', json_path.resolve())

JSON guardado en: C:\Users\izipa\Downloads\hikvision_paso_a_paso\output\hikvision_reporte.json


## Paso 12 — Generar el informe HTML

In [12]:
def generar_html(salida, ruta_salida):
    filas = ''.join(
        f"<tr><td>{ev.get('time', '')}</td><td>{ev.get('major', '')}</td>"
        f"<td>{ev.get('minor', '')}</td><td>{ev.get('name', '') or ev.get('employeeNoString', '')}</td></tr>"
        for ev in salida['eventos']
    )

    html = f"""<!DOCTYPE html>
<html lang=\"es\">
<head>
<meta charset=\"UTF-8\">
<title>Informe Hikvision</title>
<style>
body {{ font-family: Arial, sans-serif; margin: 2rem; }}
table {{ border-collapse: collapse; width: 100%; }}
th, td {{ border: 1px solid #ccc; padding: 6px 10px; text-align: left; }}
th {{ background: #f2f2f2; }}
</style>
</head>
<body>
<h1>Informe Hikvision · Consulta y reportes</h1>
<p><b>Generado:</b> {salida['consultado_en']}</p>
<p><b>Equipo:</b> {salida['equipo']['host']}:{salida['equipo']['puerto']}</p>
<p><b>Rango consultado:</b> {salida['rango']['inicio']} a {salida['rango']['fin']}</p>
<p><b>Total de eventos:</b> {salida['total_eventos']}</p>
<table>
<tr><th>Hora</th><th>Major</th><th>Minor</th><th>Nombre / Empleado</th></tr>
{filas}
</table>
</body>
</html>"""

    Path(ruta_salida).write_text(html, encoding='utf-8')

html_path = Path('output/hikvision_reporte.html')
generar_html(salida, html_path)
print('Informe HTML guardado en:', html_path.resolve())

Informe HTML guardado en: C:\Users\izipa\Downloads\hikvision_paso_a_paso\output\hikvision_reporte.html


## Paso 13 — Checklist final de validación

Repasa esta lista antes de dar la utilidad por probada contra el equipo real:

In [13]:
checklist = {
    '1. Kernel usa .venv': '.venv' in sys.executable,
    '2. Dependencias importan sin error': True,  # visto en Paso 2
    '5. Puerto responde a nivel de red': ok,
    '7. device_info obtenido': device_info is not None,
    '9. Eventos obtenidos (puede ser 0 si no hay eventos en el rango)': isinstance(eventos, list),
    '11. JSON guardado': json_path.exists(),
    '12. HTML guardado': html_path.exists(),
}

print('Resumen de validación:')
for item, estado in checklist.items():
    marca = 'OK' if estado else 'PENDIENTE / REVISAR'
    print(f'  [{marca}] {item}')

Resumen de validación:
  [OK] 1. Kernel usa .venv
  [OK] 2. Dependencias importan sin error
  [PENDIENTE / REVISAR] 5. Puerto responde a nivel de red
  [PENDIENTE / REVISAR] 7. device_info obtenido
  [OK] 9. Eventos obtenidos (puede ser 0 si no hay eventos en el rango)
  [OK] 11. JSON guardado
  [OK] 12. HTML guardado
